# Temporal holdout (A20)

Does a model trained on earlier cohorts transfer to later ones?

- **Cohorts.** An address's cohort date is its `earliest_l0_tx_time` (Unix seconds, UTC). Each gas provision tree
  takes the date of its earliest member, so trees stay whole. Cut-off: the date at which about 30 % of addresses
  fall in the late cohort; 20 % and 40 % as sensitivity checks.
- **Split.** Training and validation: a two-way stratified group split (70 / 30, by tree) of the early cohort,
  implemented here with the greedy rule of `sp._group_partition` (that private function is not called or changed).
  The training partition is upsampled to 1:1 as in `sp.make_splits`. Test: the whole late cohort.
- **Model.** LightGBM, 3 seeds, parameters from `02`; threshold from validation.
- **Reference.** The seed-42 tree-split model (`sp.make_splits`, 3 seeds), trained here on the same machine, scored
  on its own test rows that fall in the late cohort.

**Caveats** (also in the results): cohort and Sybil status are confounded (the Sybil rate differs strongly between
cohorts; see the first table); features are cumulative up to the snapshot, so late-cohort addresses have shorter
histories by construction.

**Results**: `results/20_temporal_holdout.json`.

## Setup

In [1]:
import os, platform, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.utils import resample
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
print('Platform:', PLATFORM)
DATA_DIR = './data'
FEATS = sp.FEATS
_, LGBM_P = sp.load_selected_params()
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
cohort_t = df.groupby('provision_tree')['earliest_l0_tx_time'].transform('min')   # tree takes its earliest member's date
LATE_SHARES = [0.30, 0.20, 0.40]
CUTS = {q: float(cohort_t.quantile(1 - q)) for q in LATE_SHARES}
rows = []
for q, c in CUTS.items():
    late = cohort_t >= c
    rows.append(dict(late_share_target=q, cutoff_utc=str(pd.to_datetime(c, unit='s', utc=True)),
                     late_share=float(late.mean()), early_rows=int((~late).sum()), late_rows=int(late.sum()),
                     early_sybil_rate=float(df.loc[~late, 'sybil'].mean()), late_sybil_rate=float(df.loc[late, 'sybil'].mean()),
                     early_sybils=int(df.loc[~late, 'sybil'].sum()), late_sybils=int(df.loc[late, 'sybil'].sum())))
cohorts = pd.DataFrame(rows)
print(cohorts.round(4).to_string(index=False))

Code commit: dc4d551
Platform: {'machine': 'x86_64', 'processor': 'x86_64', 'system': 'Linux', 'python': '3.11.15', 'cpu_count': 4, 'n_jobs': 4}


 late_share_target                       cutoff_utc  late_share  early_rows  late_rows  early_sybil_rate  late_sybil_rate  early_sybils  late_sybils
               0.3 2023-08-20 11:32:02.500000+00:00         0.3      304350     130436            0.0555           0.0102         16885         1326
               0.2        2023-12-17 17:14:40+00:00         0.2      347828      86958            0.0516           0.0029         17960          251
               0.4        2023-06-27 12:20:13+00:00         0.4      260871     173915            0.0511           0.0280         13337         4874


## Two-way stratified group split of the early cohort

The greedy rule of `sp._group_partition` with two partitions (70 / 30): multi-wallet trees first, largest first
(ties in random order), each into the partition with the most remaining per-class room relative to its target;
single-wallet trees then fill each partition's remaining per-class quota at random.

In [2]:
def two_way_group_partition(d, seed, fracs=(0.7, 0.3)):
    fracs = np.asarray(fracs); rng = np.random.default_rng(seed)
    y = d['sybil'].to_numpy(); n_cls = np.array([(y == 0).sum(), (y == 1).sum()])
    target = np.outer(fracs, n_cls).astype(float); quota = target.copy()
    g = d.groupby('provision_tree', sort=False)['sybil'].agg(['size', 'sum'])
    multi = g[g['size'] > 1]
    multi = multi.iloc[rng.permutation(len(multi))].sort_values('size', ascending=False, kind='stable')
    part_of = {}
    for grp, (size, n_syb) in zip(multi.index, multi[['size', 'sum']].to_numpy()):
        mix = np.array([size - n_syb, n_syb]) / n_cls
        p = int(np.argmax((quota / target) @ mix)); quota[p] -= (size - n_syb, n_syb); part_of[grp] = p
    part = np.array(d['provision_tree'].map(part_of), dtype=float)
    single = np.isnan(part)
    for c in (0, 1):
        idx = rng.permutation(np.flatnonzero(single & (y == c)))
        take = int(max(round(quota[1, c]), 0))
        part[idx[:take]] = 1; part[idx[take:]] = 0
    part = part.astype(int); ix = d.index.to_numpy()
    return ix[part == 0], ix[part == 1]

def build(tr, va, te, seed):
    X, y = df[FEATS].astype(np.float32), df['sybil'].astype(int)
    trn = pd.concat([X.loc[tr], y.loc[tr].rename('sybil')], axis=1)
    maj, mn = trn[trn['sybil'] == 0], trn[trn['sybil'] == 1]
    bal = pd.concat([maj, resample(mn, replace=True, n_samples=len(maj), random_state=seed)]).sample(frac=1, random_state=seed)
    return dict(method='temporal', X_train=bal.drop(columns='sybil').astype(np.float32), y_train=bal['sybil'],
                X_val=X.loc[va], y_val=y.loc[va], X_test=X.loc[te], y_test=y.loc[te], idx_train=tr, idx_val=va, idx_test=te)

def metrics(y, p, thr):
    y = np.asarray(y); yhat = (p >= thr).astype(int)
    return dict(n=len(y), sybils=int(y.sum()), threshold=float(thr), precision=precision_score(y, yhat, zero_division=0),
                recall=recall_score(y, yhat, zero_division=0), f1=f1_score(y, yhat, zero_division=0),
                auroc=roc_auc_score(y, p), ap=average_precision_score(y, p))

## Reference: the seed-42 tree-split model (trained here)

In [3]:
t0 = time.time()
S_ref = sp.make_splits(df, FEATS, method='group', seed=sp.SEED)
m_ref = sp.fit_lgbm(S_ref, LGBM_P)
thr_ref = sp.best_f1_threshold(S_ref['y_val'], m_ref['val'])
print(f'({time.time() - t0:.0f}s)  threshold {thr_ref:.4f}')

  LightGBM seed=42  rounds=315  47.8s


  LightGBM seed=123  rounds=298  46.6s


  LightGBM seed=456  rounds=301  48.1s


(145s)  threshold 0.5444


## Temporal holdout per cut-off

In [4]:
results, by_cat = [], []
for q, c in CUTS.items():
    t0 = time.time()
    late = cohort_t >= c
    early_df = df.loc[~late]
    tr, va = two_way_group_partition(early_df, sp.SEED)
    te = df.index[late].to_numpy()
    g = df['provision_tree']
    assert not (set(g.loc[tr]) & set(g.loc[va]) or set(g.loc[tr]) & set(g.loc[te]) or set(g.loc[va]) & set(g.loc[te]))
    S = build(tr, va, te, sp.SEED)
    m = sp.fit_lgbm(S, LGBM_P)
    thr = sp.best_f1_threshold(S['y_val'], m['val'])
    r = metrics(S['y_test'], m['test'], thr)
    # reference: the tree-split model on its own test rows in this late cohort
    ref_rows = late.loc[S_ref['idx_test']].to_numpy()
    rr = metrics(S_ref['y_test'].to_numpy()[ref_rows], m_ref['test'][ref_rows], thr_ref)
    results.append(dict(late_share_target=q, cutoff_utc=str(pd.to_datetime(c, unit='s', utc=True)),
                        train_unique=len(tr), val=len(va), train_sybil_rate=float(df.loc[tr, 'sybil'].mean()),
                        val_sybil_rate=float(df.loc[va, 'sybil'].mean()), rounds=m['rounds'],
                        **{f'temporal_{k}': v for k, v in r.items()}, **{f'reference_{k}': v for k, v in rr.items()}))
    for name, y, p, t in [('temporal', S['y_test'], m['test'], thr),
                          ('reference', S_ref['y_test'].to_numpy()[ref_rows], m_ref['test'][ref_rows], thr_ref)]:
        cat = df.loc[S['idx_test'], 'category'] if name == 'temporal' else df.loc[S_ref['idx_test'], 'category'].to_numpy()[ref_rows]
        bc = sp.metrics_by_category(y, p, cat, t)
        by_cat += [dict(late_share_target=q, model=name, **x) for x in bc.to_dict('records')]
    print(f"late {q:.0%} (from {results[-1]['cutoff_utc'][:10]}): temporal F1 {r['f1']:.4f} AP {r['ap']:.4f} AUROC {r['auroc']:.4f}; "
          f"reference on late rows F1 {rr['f1']:.4f} AP {rr['ap']:.4f} AUROC {rr['auroc']:.4f} ({time.time() - t0:.0f}s)", flush=True)
results = pd.DataFrame(results); by_cat = pd.DataFrame(by_cat)

  LightGBM seed=42  rounds=402  55.2s


  LightGBM seed=123  rounds=389  53.4s


  LightGBM seed=456  rounds=340  49.1s


late 30% (from 2023-08-20): temporal F1 0.1661 AP 0.2801 AUROC 0.8882; reference on late rows F1 0.7345 AP 0.7837 AUROC 0.9830 (162s)


  LightGBM seed=42  rounds=358  54.3s


...

late 20% (from 2023-12-17): temporal F1 0.0000 AP 0.0443 AUROC 0.7424; reference on late rows F1 0.4944 AP 0.4491 AUROC 0.9528 (178s)


  LightGBM seed=42  rounds=299  42.8s


  LightGBM seed=123  rounds=332  45.2s


  LightGBM seed=456  rounds=311  43.7s


late 40% (from 2023-06-27): temporal F1 0.2127 AP 0.2564 AUROC 0.9114; reference on late rows F1 0.7236 AP 0.8097 AUROC 0.9847 (135s)


## Summary

In [5]:
cols = ['late_share_target', 'cutoff_utc', 'temporal_sybils', 'temporal_precision', 'temporal_recall', 'temporal_f1',
        'temporal_ap', 'temporal_auroc', 'reference_sybils', 'reference_precision', 'reference_recall', 'reference_f1',
        'reference_ap', 'reference_auroc']
print(results[cols].round(4).to_string(index=False))
print()
print(by_cat[by_cat['category'].isin(['IxL', 'IxI', 'IxE'])].round(4).to_string(index=False))

 late_share_target                       cutoff_utc  temporal_sybils  temporal_precision  temporal_recall  temporal_f1  temporal_ap  temporal_auroc  reference_sybils  reference_precision  reference_recall  reference_f1  reference_ap  reference_auroc
               0.3 2023-08-20 11:32:02.500000+00:00             1326              0.7935           0.0928       0.1661       0.2801          0.8882               383               0.7762            0.6971        0.7345        0.7837           0.9830
               0.2        2023-12-17 17:14:40+00:00              251              0.0000           0.0000       0.0000       0.0443          0.7424                64               0.8800            0.3438        0.4944        0.4491           0.9528
               0.4        2023-06-27 12:20:13+00:00             4874              0.4182           0.1426       0.2127       0.2564          0.9114              1458               0.6781            0.7757        0.7236        0.8097           0.9847


## Save results

In [6]:
sp.save_results([], '20_temporal_holdout', extra=dict(
    params=LGBM_P, cohorts=cohorts.to_dict('records'), results=results.to_dict('records'), by_category=by_cat.to_dict('records'),
    reference=dict(split='seed-42 tree split, 3 seeds, trained in this notebook', threshold=thr_ref),
    caveats=['Cohort and Sybil status are confounded: the Sybil rate differs strongly between the early and late cohorts '
             '(see cohorts), so the late test set has a different prevalence; AP depends on prevalence.',
             'Features are cumulative up to the snapshot, so late-cohort addresses have shorter histories by construction.'],
    platform=PLATFORM))

Saved results/20_temporal_holdout.json
